# Credit Approval EDA
Exploratory Data Analysis of the UCI Credit Approval dataset (690 samples, 15 mixed-type features).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import seaborn as sns
from pathlib import Path

from src.utils import load_raw_data, cramers_v, CATEGORICAL_COLS, CONTINUOUS_COLS, TARGET_COL

DATA_PATH = "../data/raw/crx.data"
df = load_raw_data(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Class distribution
print("Target distribution:")
print(df[TARGET_COL].value_counts())
print(f"Approval rate: {df[TARGET_COL].mean():.1%}")

fig, ax = plt.subplots(figsize=(5, 4))
df[TARGET_COL].value_counts().plot(kind="bar", ax=ax, color=["#e74c3c","#2ecc71"])
ax.set_xticklabels(["Rejected (0)", "Approved (1)"], rotation=0)
ax.set_title("Class Distribution")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("../outputs/reports/class_distribution.png", dpi=150)
plt.show()
print("Saved class_distribution.png")

In [ ]:
# Missing values
print("Missing values per column:")
missing = df.isnull().sum()
print(missing[missing > 0])

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df.isnull().T, cbar=False, yticklabels=df.columns, ax=ax, cmap="Reds")
ax.set_title("Missing Value Map")
plt.tight_layout()
plt.savefig("../outputs/reports/missing_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Continuous feature distributions
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flatten(), CONTINUOUS_COLS):
    df[col].dropna().hist(bins=30, ax=ax, color="#3498db", edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel("")
plt.suptitle("Continuous Feature Distributions", y=1.01)
plt.tight_layout()
plt.savefig("../outputs/reports/continuous_distributions.png", dpi=150)
plt.show()

In [ ]:
# Cramér's V heatmap for categorical associations with target
cat_cols_with_target = CATEGORICAL_COLS + [TARGET_COL]
cramer_matrix = pd.DataFrame(index=cat_cols_with_target, columns=cat_cols_with_target, dtype=float)
for col1 in cat_cols_with_target:
    for col2 in cat_cols_with_target:
        try:
            cramer_matrix.loc[col1, col2] = cramers_v(
                df[col1].dropna().astype(str),
                df[col2].dropna().astype(str)
            )
        except Exception:
            cramer_matrix.loc[col1, col2] = 0.0

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cramer_matrix.astype(float), annot=True, fmt=".2f", cmap="YlOrRd", ax=ax)
ax.set_title("Cramér's V — Categorical Associations (incl. Target)")
plt.tight_layout()
plt.savefig("../outputs/reports/cramers_v_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Pearson correlation for continuous features
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[CONTINUOUS_COLS + [TARGET_COL]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Pearson Correlation — Continuous Features + Target")
plt.tight_layout()
plt.savefig("../outputs/reports/correlation_heatmap.png", dpi=150)
plt.show()

## Summary
- Dataset has 690 samples with slight class imbalance (55% rejected)
- 37 rows have missing values across 7 columns
- A2 (age), A3, A8, A11, A14, A15 are continuous
- Several categorical features show strong association with target (A9, A10, A4)
- A15 (savings) and A11 are heavily right-skewed → log1p transform warranted